# Mixed Rank + Expert Follow-Up Selection

Goal: derive a compact follow-up matrix by mixing the best findings from the expert sweep and the rank sweep.

We use:

- best expert counts: `32` and `2` from the no-CoLA expert analysis
- best configured ranks: `2` and `4` from the rank analysis
- `e2 -> k1`, because with two experts `k2` would activate all experts
- for `e32` (using 32 experts), its still not obvious which topk value to choose. Therefore i have decided to look at the previous sweeps and check for the selected approaches which topk value performed the best

In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
from scipy.stats import friedmanchisquare, wilcoxon
from statsmodels.stats.multitest import multipletests

ROOT = Path("/local/joeldag/repos/moe-study")
EXPERT_REPORT = ROOT / "outputs/pipeline_runs/moe_study_expert_sweep_pipeline_report/expert_pipeline_report.xlsx"

pd.set_option("display.max_columns", 80)


def parse_count(run_id):
  for pat in [r"_e(\d+)(?:_|$)", r"_heads(\d+)$", r"_nup(\d+)$", r"_b(\d+)$"]:
      m = re.search(pat, str(run_id))
      if m:
          return int(m.group(1))
  return np.nan


def load_expert_metrics():
  raw = pd.read_excel(EXPERT_REPORT, sheet_name="benchmark_raw")
  primary = raw[
    ((raw.benchmark == "Belebele") & (raw.metric == "acc_norm")) |
    ((raw.benchmark == "FLORES") & (raw.metric.astype(str).str.startswith("bits_per_byte "))) |
    ((raw.benchmark == "SIB200") & (raw.metric == "acc")) |
    ((raw.benchmark == "Taxi1500") & (raw.metric == "acc"))
  ].copy()
  primary = primary[~primary["is_baseline"].astype(str).str.lower().isin(["true", "1", "yes"])]
  primary["direction"] = primary.metric.astype(str).str.extract(r"(en->x|x->en)", expand=False).fillna("")
  primary["score_family"] = np.where(primary.benchmark.eq("FLORES"), "FLORES " + primary.direction, primary.benchmark)
  primary["component_count"] = primary.run_id.map(parse_count)
  primary["top_k"] = primary.run_id.astype(str).str.extract(r"_k(\d+)(?:_|$)", expand=False).astype(float)
  primary["value"] = pd.to_numeric(primary.value, errors="coerce")

  g = primary.groupby(["score_family", "task", "metric"]).value
  lo, hi = g.transform("min"), g.transform("max")
  primary["normalized_score"] = np.where(primary.better.eq("lower"), (hi - primary.value) / (hi - lo), (primary.value - lo) / (hi - lo))
  primary.loc[(hi - lo).eq(0), "normalized_score"] = 1.0

  metrics = primary.dropna(subset=["value", "normalized_score", "component_count"]).copy()
  metrics["component_count"] = metrics.component_count.astype(int)
  return metrics


metrics = load_expert_metrics()
print("rows:", len(metrics), "runs:", metrics.run_id.nunique())

rows: 2856 runs: 68


## Known Inputs

From the significance notebooks we keep these selections:

- best expert counts: `32`, `2`
- best configured ranks: `2`, `4`

Best configs by expert count:

- `32`: `moelora_r2_e32_k8`, `adamoe_1_r2_e32_k4`, `moelora_r2_e32_k4`
- `2`: `hydralora_r2_heads2`, `mov_dense_e2`, `adamoe_1_r2_e2_k1`

Best configs by configured rank:

- `2`: `moelora_tr16_e8_k2`, `mola_tr16_e8_k6`, `hydralora_r2`
- `4`: `mola_tr32_e8_k6`, `moelora_tr32_e8_k2`, `lora_r4`

## Step 1: Vary Rank For The Best Expert-Count Configs

We take the best expert-count configs and switch configured rank to `2` and `4` where the method supports rank.

`mov_dense_e2` is kept unchanged because this dense expert baseline does not expose the same LoRA-rank axis in this selection.

In [2]:
rank_variants_from_expert_winners = pd.DataFrame([
  ["expert_winner_rank_variant", "moelora_r2_e32_k8", "expert count 32 winner, configured rank 2"],
  ["expert_winner_rank_variant", "moelora_r4_e32_k8", "same config with configured rank 4"],
  ["expert_winner_rank_variant", "adamoe_1_r2_e32_k4", "expert count 32 winner, configured rank 2"],
  ["expert_winner_rank_variant", "adamoe_1_r4_e32_k4", "same config with configured rank 4"],
  ["expert_winner_rank_variant", "moelora_r2_e32_k4", "expert count 32 winner, configured rank 2"],
  ["expert_winner_rank_variant", "moelora_r4_e32_k4", "same config with configured rank 4"],
  ["expert_winner_rank_variant", "hydralora_r2_heads2", "expert count/head count 2 winner, configured rank 2"],
  ["expert_winner_rank_variant", "hydralora_r4_heads2", "same config with configured rank 4"],
  ["expert_winner_rank_variant", "mov_dense_e2", "kept unchanged"],
  ["expert_winner_rank_variant", "adamoe_1_r2_e2_k1", "expert count 2 winner, configured rank 2"],
  ["expert_winner_rank_variant", "adamoe_1_r4_e2_k1", "same config with configured rank 4"],
], columns=["source", "config", "reason"])

display(rank_variants_from_expert_winners)

,source,config,reason
0,expert_winner_rank_variant,moelora_r2_e32_k8,"expert count 32 winner, configured rank 2"
1,expert_winner_rank_variant,moelora_r4_e32_k8,same config with configured rank 4
2,expert_winner_rank_variant,adamoe_1_r2_e32_k4,"expert count 32 winner, configured rank 2"
3,expert_winner_rank_variant,adamoe_1_r4_e32_k4,same config with configured rank 4
4,expert_winner_rank_variant,moelora_r2_e32_k4,"expert count 32 winner, configured rank 2"
5,expert_winner_rank_variant,moelora_r4_e32_k4,same config with configured rank 4
6,expert_winner_rank_variant,hydralora_r2_heads2,"expert count/head count 2 winner, configured r..."
7,expert_winner_rank_variant,hydralora_r4_heads2,same config with configured rank 4
8,expert_winner_rank_variant,mov_dense_e2,kept unchanged
9,expert_winner_rank_variant,adamoe_1_r2_e2_k1,"expert count 2 winner, configured rank 2"


## Step 2: Choose Missing `e32` Top-K Values

For `e2`, we set `k1`. For `e32`, we directly check all available `moelora` and `mola` expert-sweep runs with 32 experts and any top-k value.

For each approach we use Friedman to rank all available top-k values, then Wilcoxon signed-rank for best vs second-best, with Holm correction.


In [3]:
e32_topk = metrics[
  metrics.kind.isin(["moelora", "mola"]) &
  metrics.component_count.eq(32) &
  metrics.top_k.notna()
].copy()
e32_topk["top_k"] = e32_topk.top_k.astype(int)

available_e32_topk = (
  e32_topk.groupby(["kind", "run_id", "top_k"])
  .agg(mean_score=("normalized_score", "mean"), n_rows=("normalized_score", "count"))
  .sort_values(["kind", "mean_score"], ascending=[True, False])
  .reset_index()
)

print("Available e32 top-k runs")
display(available_e32_topk)

friedman_rows = []
pairwise_rows = []

for kind, g in e32_topk.groupby("kind"):
  pivot = (
    g.groupby(["score_family", "task", "language", "metric", "top_k"])["normalized_score"]
    .mean()
    .unstack("top_k")
    .dropna()
  )

  stat, p = friedmanchisquare(*[pivot[c] for c in pivot.columns])
  avg_ranks = pivot.rank(axis=1, ascending=False).mean().sort_values()
  best, second = avg_ranks.index[:2]

  friedman_rows.append({
    "kind": kind,
    "best_top_k": int(best),
    "second_best_top_k": int(second),
    "n_pairs": len(pivot),
    "friedman_p_raw": p,
    "kendalls_w": stat / (len(pivot) * (pivot.shape[1] - 1)),
    **{f"avg_rank_k{int(k)}": avg_ranks[k] for k in avg_ranks.index},
  })

  w = wilcoxon(pivot[best], pivot[second], zero_method="pratt", alternative="greater")
  pairwise_rows.append({
    "kind": kind,
    "a": f"k{int(best)}",
    "b": f"k{int(second)}",
    "mean_a": pivot[best].mean(),
    "mean_b": pivot[second].mean(),
    "median_delta_a_minus_b": (pivot[best] - pivot[second]).median(),
    "n_pairs": len(pivot),
    "p_raw": w.pvalue,
  })

friedman_topk = pd.DataFrame(friedman_rows)
friedman_topk["p_holm"] = multipletests(friedman_topk.friedman_p_raw, method="holm")[1]
friedman_topk["significant"] = friedman_topk.p_holm < 0.05

pairwise_topk = pd.DataFrame(pairwise_rows)
pairwise_topk["p_holm"] = multipletests(pairwise_topk.p_raw, method="holm")[1]
pairwise_topk["significant"] = pairwise_topk.p_holm < 0.05

print("Friedman top-k result")
display(friedman_topk)
print("Wilcoxon best vs second-best")
display(pairwise_topk)

Available e32 top-k runs


,kind,run_id,top_k,mean_score,n_rows
0,moelora,moelora_r2_e32_k8,8,0.935823,42
1,moelora,moelora_r2_e32_k4,4,0.916821,42
2,moelora,moelora_r2_e32_k2,2,0.906169,42
3,moelora,moelora_r2_e32_k1,1,0.870652,42
4,mola,adamoe_1_r2_e32_k2,2,0.930762,42
5,mola,adamoe_1_r2_e32_k4,4,0.925381,42
6,mola,adamoe_1_r2_e32_k16,16,0.917686,42
7,mola,adamoe_1_r2_e32_k8,8,0.907511,42
8,mola,adamoe_1_r2_e32_k1,1,0.898315,42


Friedman top-k result


,kind,best_top_k,second_best_top_k,n_pairs,friedman_p_raw,kendalls_w,avg_rank_k8,avg_rank_k4,avg_rank_k1,avg_rank_k2,avg_rank_k16,p_holm,significant
0,moelora,8,4,42,0.000022,0.192246,2.00000,2.059524,2.785714,3.154762,NaN,0.000045,True
1,mola,4,2,42,0.000497,0.119107,2.72619,2.452381,3.642857,2.678571,3.5,0.000497,True


Wilcoxon best vs second-best


,kind,a,b,mean_a,mean_b,median_delta_a_minus_b,n_pairs,p_raw,p_holm,significant
0,moelora,k8,k4,0.935823,0.916821,0.000196,42,0.011656,0.023312,True
1,mola,k4,k2,0.925381,0.930762,0.000000,42,0.374890,0.374890,False


Top-k decision from the expert sweep:

- `moelora e32`: use `k8`
- `mola e32`: use `k4`

This follows the Friedman rank ordering. The Wilcoxon post-hoc table shows whether the best-vs-second difference is significant.


In [4]:
rank_winner_expert_variants = pd.DataFrame([
  ["rank_winner_expert_variant", "moelora_tr64_e32_k8", "configured rank 2, expert count 32, best moelora e32 top-k"],
  ["rank_winner_expert_variant", "moelora_tr4_e2_k1", "configured rank 2, expert count 2, k1"],
  ["rank_winner_expert_variant", "mola_tr64_e32_k4", "configured rank 2, expert count 32, best mola e32 top-k"],
  ["rank_winner_expert_variant", "mola_tr4_e2_k1", "configured rank 2, expert count 2, k1"],
  ["rank_winner_expert_variant", "hydralora_r2_heads32", "configured rank 2, head count 32"],
  ["rank_winner_expert_variant", "hydralora_r2_heads2", "configured rank 2, head count 2"],
  ["rank_winner_expert_variant", "mola_tr128_e32_k4", "configured rank 4, expert count 32, best mola e32 top-k"],
  ["rank_winner_expert_variant", "mola_tr8_e2_k1", "configured rank 4, expert count 2, k1"],
  ["rank_winner_expert_variant", "moelora_tr128_e32_k8", "configured rank 4, expert count 32, best moelora e32 top-k"],
  ["rank_winner_expert_variant", "moelora_tr8_e2_k1", "configured rank 4, expert count 2, k1"],
  ["rank_winner_expert_variant", "lora_r4", "rank 4 baseline, no expert-count axis"],
], columns=["source", "config", "reason"])

display(rank_winner_expert_variants)

,source,config,reason
0,rank_winner_expert_variant,moelora_tr64_e32_k8,"configured rank 2, expert count 32, best moelo..."
1,rank_winner_expert_variant,moelora_tr4_e2_k1,"configured rank 2, expert count 2, k1"
2,rank_winner_expert_variant,mola_tr64_e32_k4,"configured rank 2, expert count 32, best mola ..."
3,rank_winner_expert_variant,mola_tr4_e2_k1,"configured rank 2, expert count 2, k1"
4,rank_winner_expert_variant,hydralora_r2_heads32,"configured rank 2, head count 32"
5,rank_winner_expert_variant,hydralora_r2_heads2,"configured rank 2, head count 2"
6,rank_winner_expert_variant,mola_tr128_e32_k4,"configured rank 4, expert count 32, best mola ..."
7,rank_winner_expert_variant,mola_tr8_e2_k1,"configured rank 4, expert count 2, k1"
8,rank_winner_expert_variant,moelora_tr128_e32_k8,"configured rank 4, expert count 32, best moelo..."
9,rank_winner_expert_variant,moelora_tr8_e2_k1,"configured rank 4, expert count 2, k1"


## Final Proposed Mixed Follow-Up Matrix

The first tables keep the names from the source analyses. For launching, the table below maps equivalent historical names to one canonical naming style and removes semantic duplicates.

Example: `moelora_r2_e32_k8` and `moelora_tr64_e32_k8` mean the same configured-rank-2 / expert-count-32 config, because `2 x 32 = 64`.


In [5]:
canonical_name = {
  "moelora_r2_e32_k8": "moelora_tr64_e32_k8",
  "moelora_r4_e32_k8": "moelora_tr128_e32_k8",
  "moelora_r2_e32_k4": "moelora_tr64_e32_k4",
  "moelora_r4_e32_k4": "moelora_tr128_e32_k4",
  "adamoe_1_r2_e32_k4": "mola_tr64_e32_k4",
  "adamoe_1_r4_e32_k4": "mola_tr128_e32_k4",
  "adamoe_1_r2_e2_k1": "mola_tr4_e2_k1",
  "adamoe_1_r4_e2_k1": "mola_tr8_e2_k1",
}

proposed = pd.concat([rank_variants_from_expert_winners, rank_winner_expert_variants], ignore_index=True)
proposed["canonical_config"] = proposed.config.map(canonical_name).fillna(proposed.config)

canonical_proposed = (
  proposed
  .drop_duplicates(subset=["canonical_config"])
  .loc[:, ["canonical_config", "source", "config", "reason"]]
  .reset_index(drop=True)
)

print("n canonical proposed configs:", len(canonical_proposed))
display(canonical_proposed)

n canonical proposed configs: 15


,canonical_config,source,config,reason
0,moelora_tr64_e32_k8,expert_winner_rank_variant,moelora_r2_e32_k8,"expert count 32 winner, configured rank 2"
1,moelora_tr128_e32_k8,expert_winner_rank_variant,moelora_r4_e32_k8,same config with configured rank 4
2,mola_tr64_e32_k4,expert_winner_rank_variant,adamoe_1_r2_e32_k4,"expert count 32 winner, configured rank 2"
3,mola_tr128_e32_k4,expert_winner_rank_variant,adamoe_1_r4_e32_k4,same config with configured rank 4
4,moelora_tr64_e32_k4,expert_winner_rank_variant,moelora_r2_e32_k4,"expert count 32 winner, configured rank 2"
5,moelora_tr128_e32_k4,expert_winner_rank_variant,moelora_r4_e32_k4,same config with configured rank 4
6,hydralora_r2_heads2,expert_winner_rank_variant,hydralora_r2_heads2,"expert count/head count 2 winner, configured r..."
7,hydralora_r4_heads2,expert_winner_rank_variant,hydralora_r4_heads2,same config with configured rank 4
8,mov_dense_e2,expert_winner_rank_variant,mov_dense_e2,kept unchanged
9,mola_tr4_e2_k1,expert_winner_rank_variant,adamoe_1_r2_e2_k1,"expert count 2 winner, configured rank 2"
